# WaveTrader — Strategy Backtest with AI Confirmation

Run **any** registered strategy through the bar-by-bar backtest engine with the
**WaveTrader MTF model** providing directional confirmation on every signal.

**What this notebook does:**
1. Clones the repo & installs deps (Colab) or uses local checkout
2. Loads pre-processed MTF parquet data from `processed_data/test/`
3. Discovers all registered strategies automatically
4. Runs each strategy **with** and **without** AI confirmation on every pair
5. Compares performance: Win Rate, Profit Factor, Sharpe, Max DD, Return

## 1 · Clone Repository & Setup

In [ ]:
# ── Environment Detection & Repo Setup ────────────────────────────────────
import os, sys, pathlib, subprocess, shutil

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

GITHUB_URL = "https://github.com/Unseengap/wavetrader.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/phase_lm")
    DRIVE_DATA = DRIVE_ROOT / "data"
    DRIVE_CKPT = DRIVE_ROOT / "checkpoints"
    DRIVE_PROC = DRIVE_ROOT / "processed_data"
    for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_PROC]:
        d.mkdir(parents=True, exist_ok=True)

    LOCAL_ROOT = pathlib.Path("/content/phase_lm")
    if not (LOCAL_ROOT / ".git").exists():
        subprocess.run(f"git clone {GITHUB_URL} {LOCAL_ROOT}", shell=True, check=True)
    sys.path.insert(0, str(LOCAL_ROOT))

    # Symlink data & processed_data to Drive (avoid duplicating large files)
    for name, drive_dir in [("data", DRIVE_DATA), ("processed_data", DRIVE_PROC)]:
        link = LOCAL_ROOT / name
        if not link.exists():
            link.symlink_to(drive_dir)
            print(f"  Symlinked {link} → {drive_dir}")

    # Symlink checkpoints
    ckpt_link = LOCAL_ROOT / "checkpoints"
    if not ckpt_link.exists():
        ckpt_link.symlink_to(DRIVE_CKPT)
        print(f"  Symlinked {ckpt_link} → {DRIVE_CKPT}")

    # Install deps
    subprocess.run("pip install -q pyarrow torch numpy pandas flask", shell=True)
    PROJECT_ROOT = LOCAL_ROOT
else:
    # Local — assume we're already inside the repo
    PROJECT_ROOT = pathlib.Path.cwd()
    if (PROJECT_ROOT / "wavetrader").exists():
        pass
    elif (PROJECT_ROOT.parent / "wavetrader").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the phase_lm project root")
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "processed_data" / "test"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

print(f"Project root : {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}  (exists={PROCESSED_DIR.exists()})")
print(f"Checkpoints  : {CHECKPOINT_DIR} (exists={CHECKPOINT_DIR.exists()})")

In [ ]:
# ── Core Imports ──────────────────────────────────────────────────────────
import time, json, warnings
from datetime import datetime
from typing import Dict, List, Optional, Any

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", "{:.4f}".format)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} · Device: {device}")

# WaveTrader internals
from wavetrader.config import BacktestConfig, MTFConfig
from wavetrader.strategy_backtest import run_strategy_backtest
from wavetrader.strategies.registry import get_strategy_registry
from wavetrader.strategies.ai_confirmer import AIConfirmer

print("WaveTrader modules loaded ✓")

## 2 · Load Processed MTF Forex Data

Reads all `.parquet` files from `processed_data/test/` into a nested dict:
`data[pair][timeframe] → DataFrame`

In [ ]:
# ── Load all parquet files from processed_data/test/ ─────────────────────
TF_MAP = {"15m": "15min", "1h": "1h", "4h": "4h", "1d": "1d"}
PAIR_DISPLAY = {"GBPJPY": "GBP/JPY", "EURJPY": "EUR/JPY", "GBPUSD": "GBP/USD", "USDJPY": "USD/JPY"}

all_data: Dict[str, Dict[str, pd.DataFrame]] = {}  # pair → {tf → df}

for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    # Parse filename: e.g. GBPJPY_15m.parquet
    stem = f.stem  # GBPJPY_15m
    parts = stem.rsplit("_", 1)
    if len(parts) != 2:
        continue
    pair_tag, tf_short = parts
    pair = PAIR_DISPLAY.get(pair_tag, pair_tag)
    tf = TF_MAP.get(tf_short, tf_short)

    df = pd.read_parquet(f)
    if df.index.name in ("timestamp", "date", "datetime"):
        df = df.reset_index()
    df.columns = [c.strip().lower() for c in df.columns]

    # Normalize mid-price columns → open/high/low/close
    for base in ("open", "high", "low", "close"):
        if f"{base}_mid" in df.columns and base not in df.columns:
            df = df.rename(columns={f"{base}_mid": base})

    # Normalize date column
    if "date" not in df.columns:
        for alias in ("datetime", "timestamp", "time"):
            if alias in df.columns:
                df = df.rename(columns={alias: "date"})
                break
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    all_data.setdefault(pair, {})[tf] = df

# Summary
for pair, tfs in all_data.items():
    bars = {tf: len(df) for tf, df in tfs.items()}
    print(f"  {pair:8s} — {bars}")

PAIRS = sorted(all_data.keys())
print(f"\nLoaded {len(PAIRS)} pairs: {PAIRS}")

## 3 · Discover Strategies & Load AI Confirmer

Auto-discover every strategy registered in `_BUILTIN_STRATEGIES` and load the
WaveTrader MTF checkpoint for AI directional confirmation.

In [ ]:
# ── Discover all registered strategies ────────────────────────────────────
registry = get_strategy_registry()
strategies = registry.list_strategies()

print("Registered Strategies:")
print("-" * 70)
for s in strategies:
    print(f"  {s.id:30s}  {s.name:30s}  [{s.category}]  pairs={s.pairs}")

STRATEGY_IDS = [s.id for s in strategies]
print(f"\n→ {len(STRATEGY_IDS)} strategies found")

In [ ]:
# ── Load AI Confirmer (WaveTrader MTF checkpoint) ────────────────────────
def find_latest_checkpoint(prefix: str = "wavetrader_mtf") -> Optional[pathlib.Path]:
    """Find the newest checkpoint directory matching prefix."""
    candidates = sorted(
        [d for d in CHECKPOINT_DIR.iterdir()
         if d.is_dir() and d.name.startswith(prefix) and (d / "model_weights.pt").exists()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

ckpt_path = find_latest_checkpoint("wavetrader_mtf")

if ckpt_path:
    ai_confirmer = AIConfirmer(
        checkpoint_dir=str(ckpt_path),
        device=device,
        min_ai_confidence=0.55,
        min_alignment=0.40,
        strategy_weight=0.6,
    )
    print(f"AI Confirmer ready: {ckpt_path.name} on {device}")
else:
    ai_confirmer = None
    print("⚠ No WaveTrader MTF checkpoint found — AI confirmation disabled")

## 4 · Configure Backtest Engine

Standard settings: **$100 initial balance, 10% risk per trade, 20× leverage, 3-pip spread.**
Trailing stops activate at 3× risk (R) in the backtest engine.

In [ ]:
# ── Backtest Configuration ────────────────────────────────────────────────
INITIAL_BALANCE = 100.0
RISK_PER_TRADE  = 0.10

bt_config = BacktestConfig(
    initial_balance        = INITIAL_BALANCE,
    risk_per_trade         = RISK_PER_TRADE,
    leverage               = 20.0,
    spread_pips            = 3.0,
    commission_per_lot     = 0.0,
    pip_value              = 6.5,
    min_confidence         = 0.55,
    atr_halt_multiplier    = 3.0,
    drawdown_reduce_threshold = 0.10,
)

print(f"Balance: ${bt_config.initial_balance:.0f}  |  Risk: {bt_config.risk_per_trade:.0%}  "
      f"|  Leverage: {bt_config.leverage:.0f}×  |  Spread: {bt_config.spread_pips} pips  "
      f"|  Trail activate: {bt_config.trail_activate_r}R")

## 5 · Run Backtests — All Strategies × All Pairs (Raw + AI Confirmed)

Each strategy is run **twice** per pair:
- **Raw** — strategy signals only, no AI filter
- **AI** — strategy signals confirmed by WaveTrader MTF model

In [ ]:
# ── Run all backtests ─────────────────────────────────────────────────────
results_raw: List[Dict[str, Any]] = []
results_ai:  List[Dict[str, Any]] = []
equity_curves: Dict[str, Dict] = {}  # key = "strategy|pair|mode"

def run_single_backtest(strategy_id: str, pair: str, candles: Dict[str, pd.DataFrame],
                        use_ai: bool) -> Optional[Dict[str, Any]]:
    """Run one backtest and return a metrics dict, or None on failure."""
    mode = "AI" if use_ai else "Raw"
    confirmer = ai_confirmer if use_ai else None
    strategy = registry.instantiate(strategy_id)

    # Check the strategy supports this pair
    entry = registry.get(strategy_id)
    if pair not in entry.pairs:
        return None

    t0 = time.time()
    try:
        res = run_strategy_backtest(
            strategy=strategy,
            candles=candles,
            bt_config=bt_config,
            ai_confirmer=confirmer,
            pair=pair,
            verbose=False,
        )
    except Exception as e:
        print(f"    ✗ {strategy_id} | {pair} | {mode} — ERROR: {e}")
        return None
    elapsed = time.time() - t0

    roi = (res.final_balance / INITIAL_BALANCE - 1) * 100
    trail_exits = sum(1 for t in res.trades if "trail" in (t.exit_reason or "").lower())
    tp_exits = sum(1 for t in res.trades if "take" in (t.exit_reason or "").lower())

    row = {
        "strategy": strategy_id,
        "pair": pair,
        "mode": mode,
        "trades": res.total_trades,
        "win_rate": round(res.win_rate, 4),
        "profit_factor": round(res.profit_factor, 2),
        "sharpe": res.sharpe_ratio,
        "max_dd": round(res.max_drawdown, 4),
        "final_balance": round(res.final_balance, 2),
        "return_pct": round(roi, 1),
        "tp_exits": tp_exits,
        "trail_exits": trail_exits,
        "elapsed_s": round(elapsed, 1),
    }

    # Save equity curve
    key = f"{strategy_id}|{pair}|{mode}"
    equity_curves[key] = res.equity_curve

    return row


# ── Main loop ────────────────────────────────────────────────────────────
total_runs = len(STRATEGY_IDS) * len(PAIRS) * 2
run_num = 0

for sid in STRATEGY_IDS:
    entry = registry.get(sid)
    print(f"\n{'='*60}")
    print(f"Strategy: {entry.name} ({sid})")
    print(f"{'='*60}")

    for pair in PAIRS:
        if pair not in all_data:
            continue
        candles = all_data[pair]

        # ── Raw (no AI) ──
        run_num += 1
        print(f"  [{run_num}/{total_runs}] {pair} — Raw ...", end=" ", flush=True)
        row = run_single_backtest(sid, pair, candles, use_ai=False)
        if row:
            results_raw.append(row)
            print(f"{row['trades']}tr  WR={row['win_rate']:.0%}  PF={row['profit_factor']:.2f}  "
                  f"${row['final_balance']:.2f}  ({row['elapsed_s']:.0f}s)")
        else:
            print("skipped")

        # ── AI Confirmed ──
        run_num += 1
        if ai_confirmer is None:
            print(f"  [{run_num}/{total_runs}] {pair} — AI ... skipped (no checkpoint)")
            continue
        print(f"  [{run_num}/{total_runs}] {pair} — AI  ...", end=" ", flush=True)
        row = run_single_backtest(sid, pair, candles, use_ai=True)
        if row:
            results_ai.append(row)
            print(f"{row['trades']}tr  WR={row['win_rate']:.0%}  PF={row['profit_factor']:.2f}  "
                  f"${row['final_balance']:.2f}  ({row['elapsed_s']:.0f}s)")
        else:
            print("skipped")

print(f"\n✓ All backtests complete — {len(results_raw)} raw + {len(results_ai)} AI runs")

## 6 · Performance Metrics Summary

Side-by-side comparison of Raw vs AI-confirmed results.

In [ ]:
# ── Build summary DataFrames ──────────────────────────────────────────────
df_raw = pd.DataFrame(results_raw) if results_raw else pd.DataFrame()
df_ai  = pd.DataFrame(results_ai)  if results_ai  else pd.DataFrame()
df_all = pd.concat([df_raw, df_ai], ignore_index=True) if len(df_raw) + len(df_ai) > 0 else pd.DataFrame()

if len(df_all) > 0:
    display_cols = ["strategy", "pair", "mode", "trades", "win_rate", "profit_factor",
                    "sharpe", "max_dd", "final_balance", "return_pct", "tp_exits", "trail_exits"]
    print("=" * 120)
    print("FULL RESULTS TABLE")
    print("=" * 120)
    display(df_all[display_cols].style
            .format({"win_rate": "{:.1%}", "max_dd": "{:.1%}", "return_pct": "{:+.1f}%",
                      "final_balance": "${:.2f}", "sharpe": "{:.2f}", "profit_factor": "{:.2f}"})
            .background_gradient(subset=["profit_factor"], cmap="RdYlGn", vmin=0.5, vmax=2.5)
            .background_gradient(subset=["return_pct"], cmap="RdYlGn", vmin=-50, vmax=100))
else:
    print("No results to display.")

In [ ]:
# ── Per-strategy aggregated metrics ──────────────────────────────────────
if len(df_all) > 0:
    agg = (df_all
           .groupby(["strategy", "mode"])
           .agg(
               avg_wr=("win_rate", "mean"),
               avg_pf=("profit_factor", "mean"),
               avg_sharpe=("sharpe", "mean"),
               avg_dd=("max_dd", "mean"),
               total_trades=("trades", "sum"),
               avg_return=("return_pct", "mean"),
           )
           .round(3)
           .reset_index())

    print("\nAGGREGATED BY STRATEGY + MODE")
    print("-" * 90)
    display(agg.style
            .format({"avg_wr": "{:.1%}", "avg_dd": "{:.1%}", "avg_return": "{:+.1f}%",
                      "avg_pf": "{:.2f}", "avg_sharpe": "{:.2f}"})
            .background_gradient(subset=["avg_pf"], cmap="RdYlGn", vmin=0.5, vmax=2.5))

## 7 · Equity Curves — Raw vs AI Confirmed

In [ ]:
# ── Equity curve plots ────────────────────────────────────────────────────
if len(equity_curves) == 0:
    print("No equity curves to plot.")
else:
    # Group by strategy+pair, overlay Raw vs AI
    combos = set()
    for key in equity_curves:
        sid, pair, mode = key.split("|")
        combos.add((sid, pair))
    combos = sorted(combos)

    n_plots = len(combos)
    cols = min(3, n_plots)
    rows = (n_plots + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 4 * rows), squeeze=False)
    fig.suptitle("Equity Curves — Raw vs AI Confirmed", fontsize=16, fontweight="bold", y=1.02)

    for idx, (sid, pair) in enumerate(combos):
        ax = axes[idx // cols][idx % cols]

        for mode, color, ls in [("Raw", "#2196F3", "-"), ("AI", "#FF5722", "--")]:
            key = f"{sid}|{pair}|{mode}"
            if key in equity_curves:
                eq = equity_curves[key]
                ax.plot(eq, label=mode, color=color, linestyle=ls, linewidth=1.2, alpha=0.9)

        ax.set_title(f"{sid}\n{pair}", fontsize=10)
        ax.set_xlabel("Bar")
        ax.set_ylabel("Balance ($)")
        ax.axhline(INITIAL_BALANCE, color="gray", linestyle=":", linewidth=0.8)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    # Hide unused subplots
    for idx in range(n_plots, rows * cols):
        axes[idx // cols][idx % cols].set_visible(False)

    plt.tight_layout()
    plt.show()

## 8 · Compare Strategies — Bar Charts

In [ ]:
# ── Bar charts: Raw vs AI across strategies ──────────────────────────────
if len(df_all) > 0 and "Raw" in df_all["mode"].values and "AI" in df_all["mode"].values:
    metrics_to_plot = [
        ("profit_factor", "Profit Factor", "RdYlGn"),
        ("win_rate",      "Win Rate",      "RdYlGn"),
        ("return_pct",    "Return %",      "RdYlGn"),
        ("trades",        "Total Trades",  "Blues"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("Raw vs AI-Confirmed — Strategy Comparison", fontsize=14, fontweight="bold")

    for ax, (metric, title, cmap) in zip(axes.flat, metrics_to_plot):
        pivot = df_all.pivot_table(index="strategy", columns="mode", values=metric, aggfunc="mean")
        pivot = pivot.reindex(columns=["Raw", "AI"])
        pivot.plot.bar(ax=ax, color=["#2196F3", "#FF5722"], edgecolor="white", width=0.7)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel(title)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=30)

        # Add breakeven line for PF
        if metric == "profit_factor":
            ax.axhline(1.0, color="red", linestyle=":", linewidth=1, label="Breakeven")

    plt.tight_layout()
    plt.show()
elif len(df_all) > 0:
    print("Only one mode available — skipping comparison chart.")
    # Still plot single mode
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (metric, title) in zip(axes, [("profit_factor", "Profit Factor"),
                                           ("win_rate", "Win Rate"),
                                           ("return_pct", "Return %")]):
        df_all.set_index("strategy").groupby("pair")[metric].plot.bar(ax=ax, legend=True)
        ax.set_title(title)
        ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No results to plot.")

## 9 · Drawdown Analysis

In [ ]:
# ── Drawdown curves ──────────────────────────────────────────────────────
if len(equity_curves) > 0:
    combos = sorted(set(
        (k.split("|")[0], k.split("|")[1]) for k in equity_curves
    ))

    n_plots = len(combos)
    cols = min(3, n_plots)
    rows = (n_plots + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 3 * rows), squeeze=False)
    fig.suptitle("Drawdown Curves", fontsize=14, fontweight="bold", y=1.02)

    for idx, (sid, pair) in enumerate(combos):
        ax = axes[idx // cols][idx % cols]

        for mode, color in [("Raw", "#2196F3"), ("AI", "#FF5722")]:
            key = f"{sid}|{pair}|{mode}"
            if key not in equity_curves:
                continue
            eq = np.array(equity_curves[key])
            running_max = np.maximum.accumulate(eq)
            dd = (eq - running_max) / running_max * 100
            ax.fill_between(range(len(dd)), dd, 0, alpha=0.3, color=color, label=mode)
            ax.plot(dd, color=color, linewidth=0.8, alpha=0.7)

        ax.set_title(f"{sid} | {pair}", fontsize=10)
        ax.set_xlabel("Bar")
        ax.set_ylabel("Drawdown %")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    for idx in range(n_plots, rows * cols):
        axes[idx // cols][idx % cols].set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("No equity curves to plot.")

## 10 · Final Ranking — Best Strategies

Sorted by Profit Factor, showing the **AI confirmation uplift** (if any).

In [ ]:
# ── Final ranking table ───────────────────────────────────────────────────
if len(df_all) > 0:
    # Compute AI uplift where both modes exist
    ranking = df_all.copy()
    ranking = ranking.sort_values("profit_factor", ascending=False)

    # Build uplift column
    uplift_rows = []
    for (sid, pair), grp in df_all.groupby(["strategy", "pair"]):
        raw_row = grp[grp["mode"] == "Raw"]
        ai_row  = grp[grp["mode"] == "AI"]
        if len(raw_row) > 0 and len(ai_row) > 0:
            raw_pf = raw_row.iloc[0]["profit_factor"]
            ai_pf  = ai_row.iloc[0]["profit_factor"]
            uplift = ai_pf - raw_pf
            uplift_rows.append({"strategy": sid, "pair": pair, "pf_uplift": round(uplift, 2),
                                "raw_pf": raw_pf, "ai_pf": ai_pf,
                                "raw_wr": raw_row.iloc[0]["win_rate"],
                                "ai_wr": ai_row.iloc[0]["win_rate"]})

    if uplift_rows:
        df_uplift = pd.DataFrame(uplift_rows).sort_values("pf_uplift", ascending=False)
        print("\nAI CONFIRMATION UPLIFT (PF change)")
        print("-" * 80)
        display(df_uplift.style
                .format({"pf_uplift": "{:+.2f}", "raw_pf": "{:.2f}", "ai_pf": "{:.2f}",
                          "raw_wr": "{:.1%}", "ai_wr": "{:.1%}"})
                .background_gradient(subset=["pf_uplift"], cmap="RdYlGn", vmin=-0.5, vmax=0.5))

    # Overall ranking
    print("\n" + "=" * 100)
    print("FINAL RANKING (by Profit Factor)")
    print("=" * 100)
    display(ranking[["strategy", "pair", "mode", "trades", "win_rate", "profit_factor",
                      "sharpe", "max_dd", "return_pct", "final_balance"]]
            .head(20)
            .style
            .format({"win_rate": "{:.1%}", "max_dd": "{:.1%}", "return_pct": "{:+.1f}%",
                      "final_balance": "${:.2f}", "sharpe": "{:.2f}", "profit_factor": "{:.2f}"})
            .background_gradient(subset=["profit_factor"], cmap="RdYlGn", vmin=0.5, vmax=2.5))

    # Best overall
    best = ranking.iloc[0]
    print(f"\n★ BEST: {best['strategy']} on {best['pair']} ({best['mode']}) — "
          f"PF={best['profit_factor']:.2f}, WR={best['win_rate']:.1%}, "
          f"Return={best['return_pct']:+.1f}%, ${best['final_balance']:.2f}")
else:
    print("No results to rank.")